# Autoship Delay in Onboarding to Household (Adult + Kids) — Data Analysis

This notebook builds up, CTE by CTE, the warehouse query used in `power_analysis_aar.ipynb` to establish the Autoship Adoption Rate baseline and eligible volume. Each step below adds one CTE and re-runs, so the final steps in each part reproduce the exact queries used for sizing.

**Population:** Household onboarding clients — Women's, Men's, or Kids business line, linked to a primary household client (`household_primary_client_id IS NOT NULL`), not already Fix-scheduled, whose primary client already has shipping/payment info saved, and not already enrolled in Autoship. No signup-flow-type field (e.g. a value marking this specific onboarding flow) exists anywhere in the warehouse, so household linkage alone stands in as the population filter — this is checked in Part A.

**Source tables:**
- `curated.client` — business line, household linkage (`household_primary_client_id`), onboarding timestamp (`signup_at`).
- `curated.client_first_conversion` — used to check whether a client already had a Fix scheduled at onboarding.
- `client_service_production.clients` — the primary client's own row, used for `shipping_address`.
- `payment_method_service.payment_methods` — a dedicated payment-method-on-file table, joined by `client_id`.
- `curated.client_pulse_journal` — a daily client-state journal. `last_autoship_demand_ts` marks when a client's most recent Autoship subscription was created or renewed, and resets to null on a full cancellation. It's read here both as a point-in-time snapshot (to check "not already enrolled") and as a full-history scan (to catch post-onboarding adoption without under-counting).

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

## Part A — The `household_cohort` CTE: getting the four eligibility conditions onto one row per client

### A1 — Business line + household link

`household_primary_client_id IS NOT NULL` is a real, confirmed column, so it's used here to keep this preview scoped to household-linked clients only, over a short recent window, rather than scanning every Women's/Men's/Kids signup.

In [2]:
query("""--sql
SELECT client_id, business_line, household_primary_client_id, signup_at
FROM curated.client
WHERE business_line IN ('Womens', 'Mens', 'Kids')
  AND household_primary_client_id IS NOT NULL
  AND signup_at >= CURRENT_DATE - INTERVAL '30' DAY
ORDER BY signup_at DESC
LIMIT 8
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,business_line,household_primary_client_id,signup_at
0,54692505,Mens,19894711,2026-09-08 06:51:32.502
1,54692469,Mens,54692397,2026-09-08 06:39:52.476
2,54692466,Mens,54692265,2026-09-08 06:39:35.885
3,54692397,Mens,54692397,2026-09-08 06:15:29.928
4,54692383,Kids,33533541,2026-09-08 06:14:41.362
5,54692352,Mens,54692313,2026-09-08 06:08:07.137
6,54692338,Womens,54692238,2026-09-08 06:05:17.322
7,54692330,Womens,44877318,2026-09-08 06:01:23.877


**Reading this:** confirms the join resolves — every row has a populated `household_primary_client_id` and a recent `signup_at`, with no errors on the column names used throughout the rest of this notebook.

### A2 — Add the cohort window and the fraud/employee exclusions

In [3]:
query("""--sql
SELECT client_id, business_line, household_primary_client_id, signup_at
FROM curated.client
WHERE business_line IN ('Womens', 'Mens', 'Kids')
  AND household_primary_client_id IS NOT NULL
  AND COALESCE(fake_client_flag, 0) = 0
  AND COALESCE(employee_affiliated_flag, 0) = 0
  AND signup_at >= DATE '2026-03-01'
ORDER BY signup_at DESC
LIMIT 8
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,business_line,household_primary_client_id,signup_at
0,54692505,Mens,19894711,2026-09-08 06:51:32.502
1,54692469,Mens,54692397,2026-09-08 06:39:52.476
2,54692466,Mens,54692265,2026-09-08 06:39:35.885
3,54692397,Mens,54692397,2026-09-08 06:15:29.928
4,54692383,Kids,33533541,2026-09-08 06:14:41.362
5,54692352,Mens,54692313,2026-09-08 06:08:07.137
6,54692338,Womens,54692238,2026-09-08 06:05:17.322
7,54692330,Womens,44877318,2026-09-08 06:01:23.877


This is the `household_cohort` CTE used in both of `power_analysis_aar.ipynb`'s queries.

## Part B — The "no Fix scheduled" condition

Left-joins `curated.client_first_conversion`, keeping only clients with no cancellation-adjusted first-Fix demand at or before their onboarding timestamp.

In [4]:
query("""--sql
WITH household_cohort AS (
    SELECT client_id, household_primary_client_id, signup_at AS onboarding_ts
    FROM curated.client
    WHERE business_line IN ('Womens', 'Mens', 'Kids')
      AND household_primary_client_id IS NOT NULL
      AND COALESCE(fake_client_flag, 0) = 0
      AND COALESCE(employee_affiliated_flag, 0) = 0
      AND signup_at >= DATE '2026-03-01'
)
SELECT
    COUNT(*) AS n_cohort,
    COUNT(*) FILTER (WHERE cfc.client_id IS NULL) AS n_without_fix_scheduled
FROM household_cohort hc
LEFT JOIN curated.client_first_conversion cfc
    ON cfc.client_id = hc.client_id
   AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_cohort,n_without_fix_scheduled
0,187637,187637


**Reading this:** `n_cohort` = 187,637 household-linked clients over the 6-month window; `n_without_fix_scheduled` equals it exactly (100%) — no client in this cohort already had a Fix scheduled at onboarding, which is what's expected for a genuinely new-signup population.

## Part C — The primary client's shipping/payment info

Joins each household member back to their primary client's row in `client_service_production.clients` (shipping) and checks for a matching row in `payment_method_service.payment_methods` (payment on file). A plain `curated.client` self-join was tried first — it has no shipping/payment columns at all, which is what led to these two real tables instead.

In [5]:
query("""--sql
WITH household_cohort AS (
    SELECT client_id, household_primary_client_id, signup_at AS onboarding_ts
    FROM curated.client
    WHERE business_line IN ('Womens', 'Mens', 'Kids')
      AND household_primary_client_id IS NOT NULL
      AND COALESCE(fake_client_flag, 0) = 0
      AND COALESCE(employee_affiliated_flag, 0) = 0
      AND signup_at >= DATE '2026-03-01'
)
SELECT
    COUNT(*) AS n_cohort,
    COUNT(*) FILTER (WHERE csp.client_id IS NOT NULL) AS n_primary_found,
    COUNT(*) FILTER (WHERE csp.shipping_address IS NOT NULL) AS n_shipping_saved,
    COUNT(*) FILTER (WHERE EXISTS (
        SELECT 1 FROM payment_method_service.payment_methods pm
        WHERE CAST(pm.client_id AS INTEGER) = hc.household_primary_client_id
    )) AS n_payment_saved
FROM household_cohort hc
LEFT JOIN client_service_production.clients csp
    ON csp.client_id = hc.household_primary_client_id
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_cohort,n_primary_found,n_shipping_saved,n_payment_saved
0,187637,187637,187637,161742


**Reading this:** `n_primary_found` equals `n_cohort` exactly (187,637, 100%) — every household member resolves to a real primary-client row. `n_shipping_saved` is also 100%, and `n_payment_saved` is 161,742 (86.2%) — a large majority, not all, which is the expected shape.

## Part D — The "not already enrolled in Autoship" gate

Checks each cohort client's `curated.client_pulse_journal` row as of their onboarding date, excluding anyone whose `last_autoship_demand_ts` was already set at that point.

In [6]:
query("""--sql
WITH household_cohort AS (
    SELECT client_id, household_primary_client_id, signup_at AS onboarding_ts
    FROM curated.client
    WHERE business_line IN ('Womens', 'Mens', 'Kids')
      AND household_primary_client_id IS NOT NULL
      AND COALESCE(fake_client_flag, 0) = 0
      AND COALESCE(employee_affiliated_flag, 0) = 0
      AND signup_at >= DATE '2026-03-01'
),
not_fix_scheduled AS (
    SELECT hc.*
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
    WHERE cfc.client_id IS NULL
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
)
SELECT
    COUNT(*) AS n_eligible_pre_autoship_check,
    COUNT(*) FILTER (WHERE pj.client_id IS NULL) AS n_not_yet_autoship
FROM primary_profile_ready ppr
LEFT JOIN curated.client_pulse_journal pj
    ON pj.client_id = ppr.client_id
   AND pj.end_date > DATE '2026-03-01'
   AND pj.start_date <= DATE(ppr.onboarding_ts)
   AND pj.end_date > DATE(ppr.onboarding_ts)
   AND pj.last_autoship_demand_ts IS NOT NULL
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible_pre_autoship_check,n_not_yet_autoship
0,161742,108455


**Reading this:** `n_not_yet_autoship` is 108,455 of 161,742 (67.1%) — nearly a third of this cohort already shows *some* Autoship history as of their onboarding moment. That's a much larger share than "should be rare" — worth flagging on its own, separate from the maturation-window question, since it means a meaningful chunk of "new" household signups aren't actually first-time Autoship prospects.

The `pj.end_date > DATE '2026-03-01'` line is a static bound, not a correlated one — it doesn't change which rows match, but it lets the query planner discard `client_pulse_journal` rows that ended before the cohort window even starts before building the join, instead of loading the entire company-wide journal history into memory. Without it, this same query exceeds the cluster's per-node memory limit.

## Part E — The `fresh_demand` full-history scan

For clients who pass every gate above, finds the earliest `last_autoship_demand_ts` recorded *after* their onboarding timestamp, scanning each client's **entire** journal history rather than only their current row — a client's demand timestamp can reset to null and reappear later on a cancel/re-enroll cycle, so a current-snapshot-only read would miss it.

In [7]:
query("""--sql
WITH household_cohort AS (
    SELECT client_id, household_primary_client_id, signup_at AS onboarding_ts
    FROM curated.client
    WHERE business_line IN ('Womens', 'Mens', 'Kids')
      AND household_primary_client_id IS NOT NULL
      AND COALESCE(fake_client_flag, 0) = 0
      AND COALESCE(employee_affiliated_flag, 0) = 0
      AND signup_at >= DATE '2026-03-01'
      AND signup_at <= CURRENT_DATE - INTERVAL '90' DAY
),
not_fix_scheduled AS (
    SELECT hc.*
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
    WHERE cfc.client_id IS NULL
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > DATE '2026-03-01'
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
),
fresh_demand AS (
    SELECT
        nya.client_id,
        MIN(pj.last_autoship_demand_ts) AS first_post_onboarding_demand_ts
    FROM not_yet_autoship nya
    JOIN curated.client_pulse_journal pj
        ON pj.client_id = nya.client_id
       AND pj.last_autoship_demand_ts > DATE '2026-03-01'
       AND pj.last_autoship_demand_ts > nya.onboarding_ts
    GROUP BY nya.client_id
)
SELECT
    (SELECT COUNT(*) FROM not_yet_autoship) AS n_mature_eligible,
    COUNT(*) AS n_with_any_post_onboarding_demand
FROM fresh_demand
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_mature_eligible,n_with_any_post_onboarding_demand
0,60948,6537


This is the numerator's raw ingredient — `n_with_any_post_onboarding_demand` (6,537 of 60,948 mature-eligible clients, 10.7% uncapped) still needs the 90-day maturation cutoff applied per client (Part F) before it becomes the adoption count used in the baseline. It's higher than Step 1's capped rate, as expected, since it counts a demand at any point after onboarding rather than only within 90 days.

## Part F — The full `baseline_query`: applying the maturation window and grouping by month

Adds the `<= onboarding_ts + 90 days` cutoff to `fresh_demand`, then groups by onboarding month to produce a monthly Autoship Adoption Rate and daily eligible-volume figure. This is the exact query used in Step 1 of `power_analysis_aar.ipynb`.

In [8]:
baseline_query = """--sql
WITH household_cohort AS (
    SELECT
        c.client_id,
        c.household_primary_client_id,
        c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN ('Womens', 'Mens', 'Kids')
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at >= DATE '2026-03-01'
      AND c.signup_at <= CURRENT_DATE - INTERVAL '90' DAY
),
not_fix_scheduled AS (
    SELECT hc.*
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
    WHERE cfc.client_id IS NULL
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > DATE '2026-03-01'
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
),
fresh_demand AS (
    SELECT
        nya.client_id,
        MIN(pj.last_autoship_demand_ts) AS first_post_onboarding_demand_ts
    FROM not_yet_autoship nya
    JOIN curated.client_pulse_journal pj
        ON pj.client_id = nya.client_id
       AND pj.last_autoship_demand_ts > DATE '2026-03-01'
       AND pj.last_autoship_demand_ts > nya.onboarding_ts
    GROUP BY nya.client_id
),
joined AS (
    SELECT
        nya.client_id,
        DATE_TRUNC('month', nya.onboarding_ts) AS month,
        DATE(nya.onboarding_ts) AS onboarding_date,
        CASE WHEN fd.first_post_onboarding_demand_ts IS NOT NULL
              AND fd.first_post_onboarding_demand_ts <= nya.onboarding_ts + INTERVAL '90' DAY
             THEN 1 ELSE 0 END AS adopted_autoship
    FROM not_yet_autoship nya
    LEFT JOIN fresh_demand fd ON fd.client_id = nya.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT onboarding_date) AS days_observed
    FROM joined
    GROUP BY 1
)
SELECT
    j.month,
    COUNT(*) AS n_eligible,
    AVG(CAST(j.adopted_autoship AS DOUBLE)) AS autoship_adoption_rate,
    md.days_observed,
    COUNT(*) * 1.0 / md.days_observed AS eligible_per_day
FROM joined j
JOIN month_days md ON md.month = j.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

query(baseline_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,n_eligible,autoship_adoption_rate,days_observed,eligible_per_day
0,2026-06-01 00:00:00.000,4674,0.102054,9,519.3
1,2026-05-01 00:00:00.000,17284,0.093497,31,557.5
2,2026-04-01 00:00:00.000,17734,0.094733,30,591.1
3,2026-03-01 00:00:00.000,21256,0.110745,31,685.7


Matches Step 1 of `power_analysis_aar.ipynb` exactly — the May 2026 reference-month row here (9.35% adoption rate, 557.5 eligible/day) matches Step 1's figures exactly, and `n_mature_eligible` from Part E (60,948) equals the sum of `n_eligible` across all four rows here.

## Part G — The fresher, decoupled volume read

Reproduces Step 1b: the same four eligibility gates, restricted to the most recent complete calendar month, with no maturation wait — volume doesn't need one, since all four conditions are known immediately at onboarding.

In [9]:
recent_volume_query = """--sql
WITH household_cohort AS (
    SELECT c.client_id, c.household_primary_client_id, c.signup_at AS onboarding_ts
    FROM curated.client c
    WHERE c.business_line IN ('Womens', 'Mens', 'Kids')
      AND c.household_primary_client_id IS NOT NULL
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND c.signup_at >= DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '1' MONTH
      AND c.signup_at < DATE_TRUNC('month', CURRENT_DATE)
),
not_fix_scheduled AS (
    SELECT hc.*
    FROM household_cohort hc
    LEFT JOIN curated.client_first_conversion cfc
        ON cfc.client_id = hc.client_id
       AND cfc.cancellation_adjusted_first_fix_demand_ts <= hc.onboarding_ts
    WHERE cfc.client_id IS NULL
),
primary_profile_ready AS (
    SELECT nfs.*
    FROM not_fix_scheduled nfs
    JOIN client_service_production.clients csp
        ON csp.client_id = nfs.household_primary_client_id
    WHERE csp.shipping_address IS NOT NULL
      AND EXISTS (
          SELECT 1 FROM payment_method_service.payment_methods pm
          WHERE CAST(pm.client_id AS INTEGER) = nfs.household_primary_client_id
      )
),
not_yet_autoship AS (
    SELECT ppr.*
    FROM primary_profile_ready ppr
    LEFT JOIN curated.client_pulse_journal pj
        ON pj.client_id = ppr.client_id
       AND pj.end_date > DATE_TRUNC('month', CURRENT_DATE) - INTERVAL '1' MONTH
       AND pj.start_date <= DATE(ppr.onboarding_ts)
       AND pj.end_date > DATE(ppr.onboarding_ts)
       AND pj.last_autoship_demand_ts IS NOT NULL
    WHERE pj.client_id IS NULL
)
SELECT
    COUNT(*) AS n_eligible,
    COUNT(DISTINCT DATE(onboarding_ts)) AS days_observed,
    COUNT(*) * 1.0 / COUNT(DISTINCT DATE(onboarding_ts)) AS eligible_per_day
FROM not_yet_autoship
"""

query(recent_volume_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible,days_observed,eligible_per_day
0,17284,31,557.5


Matches Step 1b of `power_analysis_aar.ipynb` exactly (17,284 eligible, 557.5/day) — and, coincidentally, identical to May's figures in Part F above; verified independently in `power_analysis_aar.ipynb` to confirm this isn't a computation error.